In [2]:
import pandas as pd
from pyspark import SparkConf
from pyspark.sql import SparkSession, functions as f, Window


In [3]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.sql.legacy.timeParserPolicy": "LEGACY",
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())

In [4]:
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/07 17:38:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
int_transactions = spark.read.parquet("../data/02_intermediate/int_transactions")
int_branches = spark.read.parquet("../data/02_intermediate/int_branches")
prm_spine = spark.read.parquet("../data/03_primary/prm_spine")

In [6]:
int_branches.show(2)

+--------------------+-----------+-----------+---------+--------------+---------------+------------+--------+---------------+----------------+---------+--------+------+-------------+
|           branch_id|branch_code|branch_type|is_active|   branch_name|am_badge_number|area_manager|  region|rm_badge_number|regional_manager|longitude|latitude|  city|station_brand|
+--------------------+-----------+-----------+---------+--------------+---------------+------------+--------+---------------+----------------+---------+--------+------+-------------+
|4BC3ADA4-5793-402...|       1884|         PE|    false|TRISTAR DAMMAM|           NULL|        NULL|     PAC|           NULL|            NULL|     NULL|    NULL|DAMMAM|           PE|
|92DB0871-73E5-423...|       1583|         PE|    false|ALDRESS MAROOJ|           NULL|        NULL|CR NORTH|           NULL|            NULL|     NULL|    NULL|RIYADH|           PE|
+--------------------+-----------+-----------+---------+--------------+--------------

In [7]:
transactions_data = int_transactions.withColumn(
    "_id",
    # f.col("customer_id")
    f.concat_ws("__", "customer_id", "customer_vehicle_id")
).withColumn(
    "_observ_end_dt",
    f.last_day(f.col("transaction_dt"))
    # f.to_date(f.date_trunc("quarter", f.col("transaction_dt")))
)

In [8]:
transactions_data.select(f.countDistinct("_id")).show()
prm_spine.select(f.countDistinct("_id")).show()
prm_spine.select("_id").distinct().join(
    transactions_data.select("_id").distinct(),
    on="_id",
    how="inner"
).select(f.countDistinct("_id")).show()

25/05/07 17:39:05 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+-------------------+
|count(DISTINCT _id)|
+-------------------+
|            3436557|
+-------------------+



+-------------------+
|count(DISTINCT _id)|
+-------------------+
|            3434067|
+-------------------+



+-------------------+
|count(DISTINCT _id)|
+-------------------+
|            3434067|
+-------------------+



In [9]:
base_sales = prm_spine.join(
    transactions_data,
    on=["_id", "_observ_end_dt"],
    how="left"
)

In [15]:
base_sales.join(
    int_branches,
    on="branch_id",
    how="left"
).filter(
    f.col("transaction_id").isNotNull()
).withColumn(
    "day_of_year",
    f.dayofyear("transaction_dt")
).filter(
    f.year("transaction_dt") == 2024
).groupBy("day_of_year", "city").agg(
    f.sum("sales_amount_net").alias("net_revenue"),
).groupBy("city").agg(
    f.avg("net_revenue").alias("net_revenue"),
).orderBy(f.col("net_revenue").desc()).show(100)

+----------------+------------------+
|            city|       net_revenue|
+----------------+------------------+
|          RIYADH| 788707.3853886165|
|          JEDDAH|390828.99352274294|
|          DAMMAM|159253.72751277717|
|       AL KHOBAR|125711.50372026439|
|        AL HASSA| 84286.55391384187|
|          MADINA|   81969.726062961|
|          MAKKAH| 81548.57453672607|
|          JUBAIL| 58158.91331834816|
|           TABUK|46962.858148575804|
|            TAIF| 43370.00472281954|
|          KHAMIS| 40104.39038348596|
|        BURAYDAH|37951.287942121075|
|           YANBU| 35032.44720421518|
|           JIZAN| 32739.52995768612|
|            ABHA| 24264.60204684276|
|        AL KHARJ|22328.335745171695|
|            HAIL|21781.289482036464|
|           QATIF|17912.480668448403|
|            BAHA|15371.283920772456|
|         UNAYZAH|12070.156788074299|
|          RABIGH|11800.386133065427|
|            ARAR|11248.265674721737|
|     HAFER BATIN|10148.465980154473|
|          S